# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ma5029blp-wq/ML-flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!pip -q install duckdb huggingface_hub
from google.colab import userdata

HF_TOKEN = userdata.get("internship")
print("Token loaded successfully!" if HF_TOKEN else "Token not found.")

Token loaded successfully!


In [3]:
import duckdb

con = duckdb.connect()

con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

One row in the raw performance table represents a client-content observation for a specific report date. The expected grain is one row per **report_date + client_hash_id + content_hash_id.** The warehouse covers dates from **2025-01-27 to 2026-06-30**. A grain verification query found** 6,390 **duplicate groups, so duplicate records must be handled before aggregation or modeling. For analysis, daily observations are aggregated into client-content level features over defined time windows.

In [4]:
REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [5]:
for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name}: {n:,} rows")

dim_clients: 104 rows
dim_content: 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily: 78,835,655 rows
fact_query_90d: 2,414,248 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the daily search performance of one content item for one client. This notebook verifies the data using the mid-panel month of March 2026 (2026-03-01 to 2026-03-31).

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- visible_queries
- top_query_share

These features describe historical search performance and query characteristics. They are available before the prediction period, so they are safe to use as model inputs.

Label / Proxy:
- is_declining

The label indicates whether a content item experienced a significant decline in impressions compared with the previous period. It is created after the observation window and is not used as a feature.

Context:
- client_hash_id
- content_hash_id
- report_date

These fields identify the client, content item, and date. They are used for grouping, joining, and splitting the data but are not used as model features.

Excluded:
- month

Reason: The month column is only used to select the analysis window (March 2026). It is not a predictive feature and is therefore excluded from the model.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three verification queries were used to validate the data contract.

1. Grain check: No duplicate rows were found for the combination of report_date, client_hash_id, and content_hash_id in March 2026, confirming the expected grain.

2. Row count and date span: The selected month contains 9,841,378 rows covering the period from 2026-03-01 to 2026-03-31.

3. Availability: Filtering with gsc_data_available IS TRUE returned 3,611,061 rows, confirming the amount of data available for analysis.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# verify Grains
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


In [8]:
# Verify row count and date span
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
""").df()

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [9]:
# Availability check (IS TRUE)
con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset has several limitations.

1. Clients have different amounts of historical data, so comparisons between clients may not always be fair.

2. Some early records contain only Google Search Console (GSC) data because GA4 data was not yet available.

3. Time windows can overlap if features and labels are created from the same period, which may introduce data leakage.

4. The dataset shows what happened to content performance, but it cannot explain the real business reasons behind changes in traffic or rankings.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    client_hash_id,
    gsc_data_start,
    ga4_data_start
FROM {TABLES['dim_clients']}
ORDER BY gsc_data_start
LIMIT 10
""").df()

,client_hash_id,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,2025-02-11,2026-03-24
3,client_fef1a8f436438636,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,2025-06-07,NaT
5,client_b10cb2997d0c7c86,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,2025-06-21,2026-02-20
8,client_3197e6291363b4db,2025-06-29,2025-11-09
9,client_625b6439094e23e4,2025-07-01,2026-02-19


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.